In [1]:
import mysql.connector
import json
import pandas as pd

In [2]:
# Path to files
ROOMS_XML = "rooms.xml"
STUDENTS_XML = "students.xml"
DB_NAME = "university"

# MySQL credentials 
MYSQL_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': '123456',        # PASSWORD
    'database': 'university',   # DB name
    'charset': 'utf8mb4',
    'autocommit': True
}

CURRENT_DATE = '2025-11-14'  

In [3]:
conn = mysql.connector.connect(**MYSQL_CONFIG)
cursor = conn.cursor()

print("MySQL connected")

# Create tables (DROP IF EXISTS)
cursor.execute("""
    DROP TABLE IF EXISTS students;
    DROP TABLE IF EXISTS rooms;
""")

cursor.execute("""
    CREATE TABLE rooms (
        id   INT PRIMARY KEY,
        name VARCHAR(255) NOT NULL
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")

cursor.execute("""
    CREATE TABLE students (
        id         INT AUTO_INCREMENT PRIMARY KEY,
        name       VARCHAR(255) NOT NULL,
        birthday   DATE NOT NULL,
        sex        ENUM('M', 'F') NOT NULL,
        room_id    INT NOT NULL,
        FOREIGN KEY (room_id) REFERENCES rooms(id) ON DELETE CASCADE
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")

conn.commit()
cursor.close()
print("Tables are created in MySQL")

conn.close()

: 

In [ ]:
conn = mysql.connector.connect(**MYSQL_CONFIG)
cursor = conn.cursor()

# Upload rooms.json
with open("rooms.json", encoding="utf-8") as f:
    rooms_data = json.load(f)

# rooms_data — это список словарей: [{"id": 1, "name": "Gryffindor"}, ...]
insert_query = "INSERT INTO rooms (id, name) VALUES (%s, %s)"
data_to_insert = [(room["id"], room["name"]) for room in rooms_data]

cursor.executemany(insert_query, data_to_insert)
conn.commit()

print(f"# of rooms: {len(rooms_data)}")
pd.read_sql("SELECT * FROM rooms ORDER BY id LIMIT 10", conn)

cursor.close()
conn.close()

In [ ]:
conn = mysql.connector.connect(**MYSQL_CONFIG)
cursor = conn.cursor()

with open("students.json", encoding="utf-8") as f:
    students_data = json.load(f)

# students_data 
insert_query = """
    INSERT INTO students (name, birthday, sex, room_id)
    VALUES (%s, %s, %s, %s)
"""
data_to_insert = [
    (s["name"], s["birthday"], s["sex"], s["room"])
    for s in students_data
]

cursor.executemany(insert_query, data_to_insert)
conn.commit()

print(f"# of students: {len(students_data)}")
pd.read_sql("SELECT * FROM students LIMIT 5", conn)

cursor.close()
conn.close()

# of students: 10000


C:\Users\User\AppData\Local\Temp\ipykernel_1540\142218400.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT * FROM students LIMIT 5", conn)


In [ ]:
conn = mysql.connector.connect(**MYSQL_CONFIG)
cursor = conn.cursor()

try:
    cursor.execute("""
        CREATE INDEX idx_students_room_birthday_sex
        ON students (room_id, birthday, sex)
    """)
    print("Index created!")
except mysql.connector.Error as err:
    print("Индекс уже существует или ошибка — но это не страшно, продолжаем!")

conn.commit()
cursor.close()
conn.close()

Index created!


In [ ]:
import decimal

conn = mysql.connector.connect(**MYSQL_CONFIG)
cursor = conn.cursor(dictionary=True)   # ← ВАЖНО: dictionary=True!

# 1. Комнаты + количество студентов
cursor.execute("""
    SELECT r.name AS room, COUNT(s.id) AS students_count
    FROM rooms r
    LEFT JOIN students s ON s.room_id = r.id
    GROUP BY r.id, r.name
    ORDER BY r.name
""")
rooms = cursor.fetchall()

# 2. 5 комнат с минимальным средним возрастом
cursor.execute(f"""
    SELECT r.name AS room, ROUND(AVG({AGE_SQL}), 2) AS avg_age
    FROM rooms r
    JOIN students s ON s.room_id = r.id
    GROUP BY r.id
    ORDER BY avg_age ASC
    LIMIT 5
""")
smallest_avg_age = cursor.fetchall()

# 3. 5 комнат с максимальной разницей возраста
cursor.execute(f"""
    SELECT r.name AS room,
           MAX({AGE_SQL}) - MIN({AGE_SQL}) AS age_diff
    FROM rooms r
    JOIN students s ON s.room_id = r.id
    GROUP BY r.id
    HAVING COUNT(s.id) > 1
    ORDER BY age_diff DESC
    LIMIT 5
""")
largest_age_diff = cursor.fetchall()

# 4. Комнаты с разнополыми студентами
cursor.execute("""
    SELECT r.name AS room
    FROM rooms r
    JOIN students s ON s.room_id = r.id
    GROUP BY r.id
    HAVING COUNT(DISTINCT s.sex) > 1
    ORDER BY r.name
""")
mixed = [{"room": row["room"]} for row in cursor.fetchall()]

cursor.close()
conn.close()

# Собираем итоговый словарь
results = {
    "rooms": rooms,
    "smallest_avg_age": smallest_avg_age,
    "largest_age_diff": largest_age_diff,
    "mixed_gender_rooms": mixed
}

In [ ]:
import mysql.connector

try:
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    print("ПОДКЛЮЧЕНИЕ К MySQL: УСПЕШНО!")
    print("База данных 'university' доступна.")
    conn.close()
except Exception as e:
    print(f"ОШИБКА: {e}")

ПОДКЛЮЧЕНИЕ К MySQL: УСПЕШНО!
База данных 'university' доступна.


In [ ]:
results = {
    "rooms": q1,
    "smallest_avg_age": q2,
    "largest_age_diff": q3,
    "mixed_gender_rooms": q4
}

print("Результаты собраны")

Результаты собраны


In [ ]:
import json
from xml.etree.ElementTree import Element, SubElement, tostring
from xml.dom import minidom

# ←←← ВАЖНО: сюда должен быть уже готов словарь results ←←←
# (из предыдущей ячейки с запросами)

# Функция, которая рекурсивно превращает Decimal → float, а всё остальное оставляет как есть
def convert_decimals(obj):
    if isinstance(obj, dict):
        return {k: convert_decimals(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_decimals(item) for item in obj]
    elif isinstance(obj, decimal.Decimal):   # ← вот тут магия
        return float(obj)                    # или int(obj), если всегда целое
    else:
        return obj

# Применяем конвертацию
results_clean = convert_decimals(results)

# ------------------ ВЫВОД ------------------
format_output = "json"        # ← поменяй на "xml", если нужен XML

if format_output == "json":
    print(json.dumps(results_clean, ensure_ascii=False, indent=2))
else:
    # XML-версия
    root = Element("results")
    for section_name, items in results_clean.items():
        section = SubElement(root, section_name)
        for item in items:
            item_elem = SubElement(section, "item")
            for k, v in item.items():
                child = SubElement(item_elem, k)
                child.text = str(v)

    # Красивый вывод
    rough_string = tostring(root, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    pretty_xml = reparsed.toprettyxml(indent="  ")
    # убираем лишнюю строку <?xml version="1.0" ?> которая появляется дважды
    print('\n'.join([line for line in pretty_xml.split('\n') if line.strip()]))

{
  "rooms": [
    {
      "room": "Room #0",
      "students_count": 9
    },
    {
      "room": "Room #1",
      "students_count": 7
    },
    {
      "room": "Room #10",
      "students_count": 13
    },
    {
      "room": "Room #100",
      "students_count": 9
    },
    {
      "room": "Room #101",
      "students_count": 10
    },
    {
      "room": "Room #102",
      "students_count": 5
    },
    {
      "room": "Room #103",
      "students_count": 12
    },
    {
      "room": "Room #104",
      "students_count": 12
    },
    {
      "room": "Room #105",
      "students_count": 5
    },
    {
      "room": "Room #106",
      "students_count": 9
    },
    {
      "room": "Room #107",
      "students_count": 12
    },
    {
      "room": "Room #108",
      "students_count": 10
    },
    {
      "room": "Room #109",
      "students_count": 7
    },
    {
      "room": "Room #11",
      "students_count": 12
    },
    {
      "room": "Room #110",
      "students_count": 9
 

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# ————————————————————————
# 1. Комнаты + количество студентов
# ————————————————————————
df1 = pd.DataFrame(results["rooms"])
df1 = df1.rename(columns={"room": "Комната", "students_count": "Кол-во студентов"})
df1 = df1.sort_values("Кол-во студентов", ascending=False).reset_index(drop=True)

# ————————————————————————
# 2. 5 комнат с минимальным средним возрастом
# ————————————————————————
df2 = pd.DataFrame(results["smallest_avg_age"])
df2 = df2.rename(columns={"room": "Комната", "avg_age": "Средний возраст"})
df2["Средний возраст"] = df2["Средний возраст"].astype(float).round(2)

# ————————————————————————
# 3. 5 комнат с максимальной разницей возраста
# ————————————————————————
df3 = pd.DataFrame(results["largest_age_diff"])
df3 = df3.rename(columns={"room": "Комната", "age_diff": "Разница в возрасте (лет)"})

# ————————————————————————
# 4. Комнаты с разнополыми студентами
# ————————————————————————
df4 = pd.DataFrame(results["mixed_gender_rooms"])
df4 = df4.rename(columns={"room": "Комната (смешанные)"})

# ————————————————————————
# Красивый вывод всех таблиц подряд
# ————————————————————————
print("="*60)
print("ВСЕ КОМНАТЫ И КОЛИЧЕСТВО СТУДЕНТОВ".center(60))
print("="*60)
display(df1)

print("\n" + "="*60)
print("ТОП-5 КОМНАТ С САМЫМ МОЛОДЫМ СРЕДНИМ ВОЗРАСТОМ".center(60))
print("="*60)
display(df2)

print("\n" + "="*60)
print("ТОП-5 КОМНАТ С САМОЙ БОЛЬШОЙ РАЗНИЦЕЙ В ВОЗРАСТЕ".center(60))
print("="*60)
display(df3)

print("\n" + "="*60)
print("КОМНАТЫ, ГДЕ ЖИВУТ И ПАРНИ, И ДЕВУШКИ".center(60))
print("="*60)
display(df4)

             ВСЕ КОМНАТЫ И КОЛИЧЕСТВО СТУДЕНТОВ             


,Комната,Кол-во студентов
0,Room #73,20
1,Room #813,20
2,Room #860,19
3,Room #731,19
4,Room #905,19
...,...,...
995,Room #49,3
996,Room #154,3
997,Room #422,2
998,Room #810,2



       ТОП-5 КОМНАТ С САМЫМ МОЛОДЫМ СРЕДНИМ ВОЗРАСТОМ       


,Комната,Средний возраст
0,Room #661,12.00
1,Room #913,20.33
2,Room #111,32.00
3,Room #957,33.56
4,Room #773,35.36



      ТОП-5 КОМНАТ С САМОЙ БОЛЬШОЙ РАЗНИЦЕЙ В ВОЗРАСТЕ      


,Комната,Разница в возрасте (лет)
0,Room #83,115
1,Room #213,115
2,Room #381,115
3,Room #395,115
4,Room #486,115



           КОМНАТЫ, ГДЕ ЖИВУТ И ПАРНИ, И ДЕВУШКИ            


,Комната (смешанные)
0,Room #0
1,Room #1
2,Room #10
3,Room #100
4,Room #101
...,...
985,Room #995
986,Room #996
987,Room #997
988,Room #998


In [ ]:
from IPython.display import display, HTML
import pandas as pd

def show_dark(df, title, max_height=500):
    display(HTML(f"""
    <h3 style="
        color: #7c93e6; 
        text-align: center; 
        margin: 25px 0 15px 0; 
        font-family: 'Segoe UI', sans-serif;
        font-size: 18px;
        letter-spacing: 0.5px;
    ">{title}</h3>
    """))
    
    styled = df.style\
        .set_properties(**{
            'text-align': 'left',
            'padding': '12px',
            'font-size': '14px',
            'color': '#d4d4d4',
            'background-color': '#1e1e1e',
            'border': 'none'
        })\
        .set_table_styles([
            {'selector': 'table', 
             'props': 'width: 100%; border-collapse: collapse; margin: 0; background-color: #1e1e1e;'},
            {'selector': 'th', 
             'props': '''
                background-color: #252526; 
                color: #cccccc; 
                font-weight: 600; 
                position: sticky; 
                top: 0; 
                border-bottom: 1px solid #3e3e42;
                padding: 14px 12px;
                text-align: left;
             '''},
            {'selector': 'td', 
             'props': '''
                border-bottom: 1px solid #2d2d30;
                padding: 10px 12px;
             '''},
            {'selector': 'tr:hover', 
             'props': 'background-color: #2a2a2a !important;'}
        ])

    display(HTML(f"""
    <div style="
        background-color: #1e1e1e;
        border: 1px solid #3e3e42;
        border-radius: 10px;
        overflow: hidden;
        box-shadow: 0 4px 20px rgba(0,0,0,0.4);
        max-height: {max_height}px;
        overflow-y: auto;
    ">
        {styled.to_html()}
    </div>
    """))

# Запускаем — тёмная тема как в VS Code!
show_dark(df1, "Все комнаты и количество студентов")
show_dark(df2, "ТОП-5 комнат с самым молодым средним возрастом")
show_dark(df3, "ТОП-5 комнат с самой большой разницей в возрасте")
show_dark(df4, "Комнаты со смешанным проживанием")

,Комната,Кол-во студентов
0,Room #73,20
1,Room #813,20
2,Room #860,19
3,Room #731,19
4,Room #905,19
5,Room #197,19
6,Room #506,18
7,Room #326,18
8,Room #448,18
9,Room #359,18


,Комната,Средний возраст
0,Room #661,12.000000
1,Room #913,20.330000
2,Room #111,32.000000
3,Room #957,33.560000
4,Room #773,35.360000


,Комната,Разница в возрасте (лет)
0,Room #83,115
1,Room #213,115
2,Room #381,115
3,Room #395,115
4,Room #486,115


,Комната (смешанные)
0,Room #0
1,Room #1
2,Room #10
3,Room #100
4,Room #101
5,Room #102
6,Room #103
7,Room #104
8,Room #105
9,Room #106


In [ ]:
import pandas as pd
import mysql.connector
from IPython.display import display, HTML

# Подключаемся к базе
conn = mysql.connector.connect(**MYSQL_CONFIG)

# Запрос: все жители комнаты 486
query = """
SELECT 
    s.name AS имя,
    s.birthday AS день_рождения,
    TIMESTAMPDIFF(YEAR, s.birthday, '2025-11-17')
        - (DATE_FORMAT(s.birthday, '%m-%d') > '11-17') AS возраст,
    s.sex AS пол,
    r.name AS комната
FROM students s
JOIN rooms r ON s.room_id = r.id
WHERE r.id = 83
ORDER BY s.birthday ASC
"""

df_room_486 = pd.read_sql(query, conn)
conn.close()

# Проверяем, есть ли вообще такая комната
if df_room_486.empty:
    display(HTML("<h3 style='color:#e74c3c; text-align:center;'>Комната 486 не найдена или пуста</h3>"))
else:
    # Красивый вывод в тёмной теме
    show_dark(df_room_486.reset_index(drop=True), 
              f"Жители комнаты 486 — {len(df_room_486)} человек")

C:\Users\User\AppData\Local\Temp\ipykernel_1540\2102438567.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_room_486 = pd.read_sql(query, conn)


,имя,день_рождения,возраст,пол,комната
0,Tiffany Frazier,1903-11-24,120,M,Room #83
1,Cesar Floyd,1905-02-19,120,F,Room #83
2,Alison Campos,1908-10-27,117,M,Room #83
3,Damon Griffin,1911-01-03,114,M,Room #83
4,Timothy Smith,1932-07-22,93,M,Room #83
5,Alicia Gaines,1968-03-21,57,M,Room #83
6,Tracy Hernandez,1983-10-19,42,M,Room #83
7,Kimberly Mccarthy,1997-04-19,28,M,Room #83
8,Clinton Vasquez,2003-12-19,20,M,Room #83
9,Sherry Smith,2015-11-15,10,F,Room #83
